# BioTutor — fine-tune **Qwen3-4B** with Unsloth

A big step up from TinyLlama-1.1B: more knowledge, better reasoning, far fewer made-up
answers, and good Greek too. Unsloth makes it fit on the **free Colab T4** and removes the
old bitsandbytes headaches.

## Before you start
1. **Runtime → Change runtime type → GPU (T4)** → Save.
2. Have your `bioinfo_qa.jsonl` ready to upload (fields: `instruction`, `response`).
3. Run the cells in order (▶️). Full run ≈ 15–25 min.

Outputs at the end: a **LoRA adapter** (`biotutor-qwen3-lora.zip`, for the FastAPI backend
on GPU) and optionally a **GGUF Q4** file (small + fast on CPU, ideal for the Oracle server).

### 1 — Install Unsloth

In [ ]:
%%capture
!pip install unsloth

### 2 — Load Qwen3-4B in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = max_seq_length,
    load_in_4bit  = True,   # 4-bit => fits the free T4 (16GB)
)
print("Loaded Qwen3-4B on", torch.cuda.get_device_name(0))

### 3 — Attach a LoRA adapter (only these small matrices get trained)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 4 — Upload your data and format it for Qwen
The **Choose Files** button appears — pick `bioinfo_qa.jsonl`. We validate it, drop
duplicates, and wrap each pair in Qwen's chat format automatically.

In [ ]:
from google.colab import files
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are BioTutor, an expert tutor in bioinformatics, biology, and machine learning. "
    "You give thorough, detailed explanations with concrete examples, and you stay detailed "
    "even when the question is short."
)

up = files.upload()                 # choose bioinfo_qa.jsonl
path = list(up.keys())[0]

rows, seen = [], set()
for line in open(path, encoding="utf-8"):
    line = line.strip()
    if not line:
        continue
    rec  = json.loads(line)
    instr = str(rec.get("instruction", "")).strip()
    resp  = str(rec.get("response", "")).strip()
    if not instr or not resp:
        continue
    k = instr.lower()
    if k in seen:
        continue
    seen.add(k)
    rows.append({"instruction": instr, "response": resp})
print(f"{len(rows)} valid Q&A pairs")

def to_text(ex):
    msgs = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = Dataset.from_list(rows).map(to_text)
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]
print("train:", len(train_ds), "| val:", len(val_ds))
print("\n--- one formatted example ---\n")
print(train_ds[0]["text"][:700])

### 5 — Train
`train_on_responses_only` masks the question/system tokens, so the model is graded only on
producing the **answer** — this is what fixed the old "loss = 0" / rambling problems.
Adjust `num_train_epochs` (3–5) if answers under/over-fit.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

trainer.train()

### 6 — Quick test

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=400,
                         do_sample=False, repetition_penalty=1.1)
    print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

ask("What is GC content and why does it matter?")
print("\n" + "="*70 + "\n")
ask("What is an ORF?")

### 7 — Save the LoRA adapter (for the FastAPI backend on GPU)
Downloads `biotutor-qwen3-lora.zip`. Unzip it into `backend/biotutor-qwen3-lora/`.

In [ ]:
model.save_pretrained("biotutor-qwen3-lora")
tokenizer.save_pretrained("biotutor-qwen3-lora")
!zip -r -q biotutor-qwen3-lora.zip biotutor-qwen3-lora
from google.colab import files
files.download("biotutor-qwen3-lora.zip")

### 8 — (Optional) Export a GGUF Q4 for CPU deployment
A ~2.5GB quantised file that runs **fast on a CPU** via llama.cpp / Ollama — perfect for the
Oracle free server (a 4B model in plain PyTorch on CPU is too slow/heavy). This cell takes a
few minutes. Skip it if you only run on GPU.

In [ ]:
model.save_pretrained_gguf("biotutor-gguf", tokenizer, quantization_method="q4_k_m")
!ls -lh biotutor-gguf/*.gguf
from google.colab import files
import glob
files.download(glob.glob("biotutor-gguf/*.gguf")[0])